# Try it yourself

**Key idea.** Each exercise asks you to predict first and measure second, the workflow to keep for real code.
1. Print np.arange(0, 2, 0.3) and np.linspace(0, 1.8, 7) with their lengths and last elements, and explain why those differ.
2. Estimate pi by Monte Carlo: draw one million (x, y) points in the unit square with a seeded generator, and take four times the fraction with x**2 + y**2 <= 1. Write it with a loop, then vectorized; report the speedup.
3. Without running them, predict the result shape (or “error”) for each pair, then check with np.broadcast_shapes: (5, 4) and (4,); (5, 4) and (5,); (5, 1) and (1, 6); (3, 1, 4) and (2, 4); (8,) and (7,).
4. Reproduce the wrong-versus-right example with 1,000 random labels and predictions and print resid_wrong.nbytes. Then repeat the thought experiment for 100,000 samples, and report the gigabytes the mistake would ask for.

In [3]:
#1
import numpy as np
np1 = np.arange(0, 2, 0.3)
np2 = np.linspace(0, 1.8, 7)
print(np1, np1.shape, np1.size)
print(np2, np2.shape, np2.size)

[0.  0.3 0.6 0.9 1.2 1.5 1.8] (7,) 7
[0.  0.3 0.6 0.9 1.2 1.5 1.8] (7,) 7


In [4]:
#2
import time
import numpy as np

def estimate_pi_loop(n_points=1000000, seed=42):
    """Estimates Pi using a traditional Python loop."""
    rng = np.random.default_rng(seed)
    inside_count = 0

    start_time = time.perf_counter()
    for _ in range(n_points):
        x = rng.random()
        y = rng.random()
        if x**2 + y**2 <= 1.0:
            inside_count += 1
    end_time = time.perf_counter()

    pi_estimate = 4 * inside_count / n_points
    execution_time = end_time - start_time
    return pi_estimate, execution_time

def estimate_pi_vectorized(n_points=1000000, seed=42):
    """Estimates Pi using NumPy vectorization."""
    rng = np.random.default_rng(seed)

    start_time = time.perf_counter()
    # Draw all (x, y) coordinates at once in an (N, 2) array
    points = rng.random((n_points, 2))
    # Check condition on all elements simultaneously
    inside_mask = points[:, 0] ** 2 + points[:, 1] ** 2 <= 1.0
    inside_count = np.sum(inside_mask)
    end_time = time.perf_counter()

    pi_estimate = 4 * inside_count / n_points
    execution_time = end_time - start_time
    return pi_estimate, execution_time


# Run benchmarks
N = 1000000
pi_loop, time_loop = estimate_pi_loop(N)
pi_vec, time_vec = estimate_pi_vectorized(N)
speedup = time_loop / time_vec

print(f"Loop Approach:       Pi = {pi_loop:.5f} | Time = {time_loop:.4f}s")
print(f"Vectorized Approach: Pi = {pi_vec:.5f} | Time = {time_vec:.4f}s")
print(f"Performance Speedup: {speedup:.2f}x faster")


Loop Approach:       Pi = 3.14298 | Time = 0.5784s
Vectorized Approach: Pi = 3.14298 | Time = 0.0106s
Performance Speedup: 54.71x faster


In [5]:
#3
# Without running them, predict the result shape (or “error”) for each pair, 
# then check with np.broadcast_shapes:
#(5, 4) and (4,); (5, 4) and (5,); (5, 1) and (1, 6); (3, 1, 4) and (2, 4); (8,) and (7,).

#  5,4 / error / 5,6 / 3,2,4 / error

import numpy as np

pairs = [((5, 4), (4,)),
         ((5, 4), (5,)),
         ((5, 1), (1, 6)),
         ((3, 1, 4), (2, 4)),
         ((8,), (7,))]

for sa, sb in pairs:
    try:
        out = np.broadcast_shapes(sa, sb)
    except ValueError:
        out = "ValueError"
    print(f"{str(sa):10s} with {str(sb):8s} -> {out}")


(5, 4)     with (4,)     -> (5, 4)
(5, 4)     with (5,)     -> ValueError
(5, 1)     with (1, 6)   -> (5, 6)
(3, 1, 4)  with (2, 4)   -> (3, 2, 4)
(8,)       with (7,)     -> ValueError


In [18]:
#4

import numpy as np

rng = np.random.default_rng(seed=5)

y_true = rng.integers(0, 200, size=1000)          # shape (4,)
y_pred = rng.random((1000, 1))*200   # shape (4, 1)

# Wrong: (4, 1) - (4,) broadcasts to (4, 4)
resid_wrong = y_pred - y_true
print(resid_wrong.shape)                    # (4, 4)
print(round(float(np.mean(resid_wrong ** 2)), 4))   # 10.8125

# Right: make both operands the same 1-D shape first
resid_right = y_pred.ravel() - y_true
print(resid_right.shape)                    # (4,)
print(round(float(np.mean(resid_right ** 2)), 4))   # 0.0625
print(resid_wrong.nbytes) 

# values for 1000, seed = 5.
# (1000, 1000)
# 6672.7926
# (1000,)
# 6696.2782
# 8000000

# values for 100000
# Unable to allocate 74.5 GiB for an array with shape (100000, 100000) and data type float64

(1000, 1000)
6672.7926
(1000,)
6696.2782
8000000
